# Treinamento temporal dos modelos

- Treino final: 2014–2019 (7.723.448 casos)
- Validação, Optuna e early stopping: 2020 (1.331.664 casos)
- Teste lacrado: 2021 (não é carregado neste notebook)

A MLP ajusta encoder e medianas somente no treino. XGBoost roda na GPU (`device="cuda"`) e LightGBM em CPU; ambos preservam `NaN` nativamente. O fit final usa todo o treino; o Optuna afina numa amostra estratificada.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('../..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from dengue_pipeline.paths import TRAIN_YEARS, VALIDATION_YEARS, TEST_YEARS


N_TRIALS            = 150                  # trials de Optuna (XGBoost/LightGBM)
TUNING_SAMPLE_SIZE  = 250_000             # amostra estratificada para o tuning
MAX_EPOCHS          = 150                 # teto de épocas da MLP (early stopping corta antes)

MLP_HIDDEN          = "1024,512,256,128"  # larguras das camadas ocultas (vírgula separa)
MLP_LEARNING_RATE   = 1e-3
MLP_BATCH_SIZE      = 16_384
MLP_DROPOUT         = 0.2
MLP_PATIENCE        = 10

DEVICE              = "cuda"              # MLP e XGBoost: "cuda" ou "cpu"
LGBM_DEVICE         = "cpu"               # LightGBM: "cpu" ou "gpu" (GPU nem sempre é mais rápido)

print("Split temporal:", TRAIN_YEARS, "| val", VALIDATION_YEARS, "| teste", TEST_YEARS)

In [ ]:
# Monta o comando com as variáveis do cell de config e dispara o treino.
cmd = (
    "../../scripts/train_models.py "
    f"--n-trials {N_TRIALS} --tuning-sample-size {TUNING_SAMPLE_SIZE} --max-epochs {MAX_EPOCHS} "
    f"--mlp-hidden {MLP_HIDDEN} --mlp-learning-rate {MLP_LEARNING_RATE} "
    f"--mlp-batch-size {MLP_BATCH_SIZE} --mlp-dropout {MLP_DROPOUT} --mlp-patience {MLP_PATIENCE} "
    f"--device {DEVICE} --lgbm-device {LGBM_DEVICE}"
)
print("Executando:", cmd)
get_ipython().run_line_magic("run", cmd)